In [ ]:
import time
import requests
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager


BASE_URL     = "https://kolesa.kz"
PRICE_FROM   = 60_000_000
LIST_WAIT    = 15
DETAIL_WAIT  = 15
PAGE_SLEEP   = (1.5, 2.5)
CARD_SLEEP   = (2.0, 3.5)
SCROLL_SLEEP = 0.8

CITIES = {
    "all": "",
    # "almaty":      "almaty",
    # "astana":      "astana",
    # "shymkent":    "shymkent",
    # "karaganda":   "karaganda",
    # "aktobe":      "aktobe",
    # "taraz":       "taraz",
    # "pavlodar":    "pavlodar",
    # "oskemen":     "oskemen",
    # "semey":       "semey",
    # "atyrau":      "atyrau",
    # "kyzylorda":   "kyzylorda",
    # "oral":        "oral",
    # "kostanay":    "kostanay",
    # "petropavl":   "petropavl",
    # "aktau":       "aktau",
    # "turkestan":   "turkestan",
    # "temirtau":    "temirtau",
    # "taldykorgan": "taldykorgan",
}


MAX_PAGES = int(input("MAX PAGES: "))


def check_website_status(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            print(f"[Requests] Сайт {url} доступен (Status: 200)")
        else:
            print(f"[Requests] Сайт вернул статус: {response.status_code}")
    except Exception as e:
        print(f"[Requests] Не удалось подключиться: {e}")

check_website_status(BASE_URL)


def build_driver():
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    options.add_argument("--accept-language=ru-RU,ru;q=0.9")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    driver.set_window_size(1440, 900)
    return driver


def build_list_url(city_slug: str, page: int) -> str:
    city_part = f"{city_slug}/" if city_slug else ""
    return f"{BASE_URL}/cars/{city_part}?price[from]={PRICE_FROM}&page={page}"


def scroll_page(driver):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 3);")
    time.sleep(SCROLL_SLEEP)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.66);")
    time.sleep(SCROLL_SLEEP)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(SCROLL_SLEEP)


def get_text(tag, default=None): # Clear field extraction (and other get_text tags)
    return tag.get_text(strip=True) if tag else default


EMPTY_DETAIL = {
    "down_payment": None, "mean_price": None,  "generation": None,
    "bodywork": None,     "mileage": None,     "transmission": None,
    "drive": None,        "wheel": None,       "color": None,
    "engine_volume": None, "customs_clearance_in_kz": None,
}

def wait_for_dynamic_content(driver):
    try:
        WebDriverWait(driver, 10).until_not(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, ".average-price-container.is-loading")
            )
        )
    except Exception:
        pass


def get_car_detail(driver, url: str) -> dict:
    try:
        driver.get(url)
        WebDriverWait(driver, DETAIL_WAIT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".offer__parameters"))
        )
    except Exception:
        return EMPTY_DETAIL.copy()

    driver.execute_script("window.scrollTo(0, 400);")
    time.sleep(1.0)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
    time.sleep(1.0)

    wait_for_dynamic_content(driver)
    time.sleep(0.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    mean_price = None

    try:
        import re, json
        page_src = driver.page_source
        match = re.search(r'window\.digitalData\s*=\s*(\{.*?\});', page_src, re.DOTALL)
        if match:
            data = json.loads(match.group(1))
            avg = data.get("product", {}).get("attributes", {}).get("avgPrice")
            if avg:
                mean_price = f"{int(avg):,}".replace(",", " ") + " ₸"
    except Exception:
        pass

    if not mean_price:
        for sel in [
            ".average-price__value",
            ".price-info__value",
            ".js__average-price .price__value",
            ".average-price-container .price",
        ]:
            tag = soup.select_one(sel)
            if tag:
                mean_price = get_text(tag)
                break

    down_payment = None
    for sel in [
        ".credit__amount",
        ".credit-block__initial-fee-value",
        ".credit__down-payment",
        "[class*='credit'][class*='amount']",
    ]:
        tag = soup.select_one(sel)
        if tag:
            down_payment = get_text(tag)
            break

    def get_param(title_attr: str):
        tag = soup.find("dt", title=title_attr)
        if tag:
            dd = tag.find_next_sibling("dd")
            return get_text(dd)

        for dt in soup.find_all("dt"):
            span = dt.find("span")
            text = get_text(span) if span else get_text(dt)
            if text and title_attr.lower() == text.lower():
                dd = dt.find_next_sibling("dd")
                return get_text(dd)

        return None


    offer_price_tag = soup.select_one(".offer__price")
    offer_price = None
    if offer_price_tag:
        raw = offer_price_tag.get_text(" ", strip=True)
        offer_price = raw

    return {
        "down_payment":            down_payment,
        "mean_price":              mean_price,
        "offer_price":             offer_price,
        "generation":              get_param("Поколение"),
        "bodywork":                get_param("Кузов"),
        "engine_volume":           get_param("Объем двигателя, л"),
        "mileage":                 get_param("Пробег"),
        "transmission":            get_param("Коробка передач"),
        "drive":                   get_param("Привод"),
        "wheel":                   get_param("Руль"),
        "color":                   get_param("Цвет"),
        "customs_clearance_in_kz": get_param("Растаможен в Казахстане"),
    }



def parse_list_page(soup: BeautifulSoup) -> list:
    cards = []
    items = soup.find_all("div", class_="a-list__item")

    for div in items:
        link_tag = div.find("a", class_="a-card__link")

        title = None

        h5_title = div.find("h5", class_="a-card__title")
        if h5_title:
            a_inside = h5_title.find("a", class_="a-card__link")
            title = get_text(a_inside) if a_inside else get_text(h5_title)

        if not title:
            advert_title = div.find(attrs={"data-test": "advert-title"})
            if advert_title:
                title = get_text(advert_title)

        if not title and link_tag:
            title = link_tag.get("title", "").strip() or None

        if not title and link_tag:
            title = get_text(link_tag)

        if not title:
            for htag in ["h5", "h2", "h3", "h4"]:
                h = div.find(htag)
                if h:
                    title = get_text(h)
                    break

        year = None
        year_span = div.find("span", class_="year")
        if year_span:
            year = get_text(year_span)

        href = link_tag.get("href") if link_tag else None
        if href and href.startswith("/"):
            href = BASE_URL + href

        price_tag = (
            div.find("span", class_="a-card__price") or
            div.find("div",  class_="a-card__price")
        )

        month_tag = div.find("div", class_="month-payment__amount")

        label_tag = div.find("span", class_="a-label__tag")

        params = div.find_all("span", class_="a-card__param")
        city_val = get_text(params[0]) if len(params) > 0 else None
        date_val = get_text(params[1]) if len(params) > 1 else None

        if not date_val:
            date_tag = div.find("span", class_=lambda c: c and "date" in c)
            date_val = get_text(date_tag)

        views_tag = div.find("span", class_="a-card__views")

        label_texts = [
            get_text(lbl)
            for lbl in div.find_all("span", class_="a-label__text")
            if get_text(lbl)
        ]
        is_verified_seller = any("Проверенный" in t for t in label_texts)
        is_from_dealer     = any("дилера" in t.lower() for t in label_texts)
        is_official_dealer = any("Официальный" in t for t in label_texts)

        cards.append({
            "title":              title,
            "year":               year,
            "link":               href,
            "price":              get_text(price_tag),
            "monthly_payment":    get_text(month_tag),
            "down_payment_pct":   get_text(label_tag),
            "city":               city_val,
            "date":               date_val,
            "views":              get_text(views_tag),
            "is_verified_seller": is_verified_seller,
            "is_from_dealer":     is_from_dealer,
            "is_official_dealer": is_official_dealer,
        })

    return cards


def scrape(city_slug: str = "", max_pages=None) -> pd.DataFrame:
    driver = build_driver()

    driver.get(BASE_URL)
    time.sleep(random.uniform(3, 5))

    all_rows = []
    page = 1

    while True:   # Pagination handling
        if max_pages and page > max_pages:
            print(f"\n  [STOP] Достигнут лимит страниц: {max_pages}")
            break

        url = build_list_url(city_slug, page)
        print(f"\n{'─'*60}")
        print(f"  Страница {page}: {url}")
        print(f"{'─'*60}")

        # Error handling
        try:
            driver.get(url)
            WebDriverWait(driver, LIST_WAIT).until(
                EC.presence_of_element_located((By.CLASS_NAME, "a-list__item"))
            )
        except Exception as e:
            print(f"  [STOP] Список не загрузился: {e}")
            break

        scroll_page(driver)

        soup  = BeautifulSoup(driver.page_source, "html.parser")
        cards = parse_list_page(soup)

        if not cards:
            print("  [STOP] Карточек не найдено — конец листинга.")
            break

        if soup.find(class_=lambda c: c and "no-result" in c):
            print("  [STOP] Сайт вернул 'нет результатов'.")
            break

        print(f"  Найдено карточек: {len(cards)}")

        for idx, card in enumerate(cards, 1):
            detail = EMPTY_DETAIL.copy()

            if card["link"]:
                try:
                    detail = get_car_detail(driver, card["link"])
                    print(
                        f"    [{idx:02d}/{len(cards)}] ✓  "
                        f"{card['title'] or '???'}  |  "
                        f"цена: {card['price']}  |  "
                        f"средняя: {detail.get('mean_price')}  |  "
                        f"поколение: {detail.get('generation')}"
                    )
                except Exception as e:
                    print(f"    [{idx:02d}/{len(cards)}] ✗  Ошибка: {e}")
            else:
                print(f"    [{idx:02d}/{len(cards)}] ✗  Нет ссылки")

            all_rows.append({**card, **detail})
            time.sleep(random.uniform(*CARD_SLEEP))

        next_btn = soup.find("a", class_=lambda c: c and "next" in c.lower())
        if not next_btn:
            pagination = soup.find("ul", class_=lambda c: c and "pager" in c.lower())
            if pagination:
                next_li = pagination.find("li", class_=lambda c: c and "next" in c.lower())
                if not next_li:
                    print("\n  [STOP] Нет следующей страницы.")
                    break
            else:
                print("\n  [STOP] Пагинация не найдена — конец.")
                break

        page += 1
        time.sleep(random.uniform(*PAGE_SLEEP))

    driver.quit()
    return pd.DataFrame(all_rows)


if __name__ == "__main__":
    frames = []

    for city_name, city_slug in CITIES.items():
        print(f"\n{'═'*60}")
        print(f"  ГОРОД: {city_name.upper()}")
        print(f"{'═'*60}")

        df_city = scrape(city_slug, max_pages=MAX_PAGES)
        df_city["city_filter"] = city_name
        frames.append(df_city)

        df_city.to_csv(f"kolesa_{city_name}.csv", index=False, encoding="utf-8-sig")
        print(f"\n  Сохранено: kolesa_{city_name}.csv  ({len(df_city)} строк)")

    df_all = pd.concat(frames, ignore_index=True)
    df_all.drop_duplicates(subset=["link"], keep="first", inplace=True)

    out = "kolesa_all_cities.csv"
    df_all.to_csv(out, index=False, encoding="utf-8-sig")  # Raw dataset saved as CSV

    print(f"\n{'═'*60}")
    print(f"  ГОТОВО. Итого: {len(df_all)} уникальных объявлений")
    print(f"  Файл: {out}")
    print(f"{'═'*60}")
    print(df_all[["title", "price", "mean_price", "generation",
                  "bodywork", "mileage", "city"]].head(10).to_string())